In [36]:
import pandas as pd

it = pd.read_csv("youtube_redirect_results_it.csv")


In [37]:
it.head(5)

,video_id,verdict,reasoning,final_url
0,YkE11u3wqcY,shorts,kept /shorts route,https://www.youtube.com/shorts/YkE11u3wqcY
1,NSxpdlnn9wo,shorts,kept /shorts route,https://www.youtube.com/shorts/NSxpdlnn9wo
2,0mRKMcTFPAY,shorts,kept /shorts route,https://www.youtube.com/shorts/0mRKMcTFPAY
3,mBu4qOdH8aI,longform,redirected to /watch,https://www.youtube.com/watch?v=mBu4qOdH8aI
4,6D2ueCdbbes,longform,redirected to /watch,https://www.youtube.com/watch?v=6D2ueCdbbes


In [38]:
it.value_counts("verdict")

verdict
longform    5958
shorts      2473
Name: count, dtype: int64

In [39]:
error_rows = it[it["verdict"] == "error"]

display(error_rows["reasoning"].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str

## food csv


In [40]:
food = pd.read_csv("youtube_redirect_results_food.csv")
food.head(5)

,video_id,verdict,reasoning,final_url
0,gVdoQQI53HU,longform,redirected to /watch,https://www.youtube.com/watch?v=gVdoQQI53HU&pp...
1,FuZKyvWuWP8,shorts,kept /shorts route,https://www.youtube.com/shorts/FuZKyvWuWP8
2,_1T0CAWGsPE,shorts,kept /shorts route,https://www.youtube.com/shorts/_1T0CAWGsPE
3,zrhBNe2EMNY,longform,redirected to /watch,https://www.youtube.com/watch?v=zrhBNe2EMNY
4,kgajE44zZfo,shorts,kept /shorts route,https://www.youtube.com/shorts/kgajE44zZfo


In [41]:
food.value_counts("verdict")

verdict
shorts      5534
longform    5316
error          2
Name: count, dtype: int64

In [42]:
error_rows_food = food[food["verdict"] == "error"]

display(error_rows_food["reasoning"].value_counts())

reasoning
video is missing, deleted, private, or unavailable    2
Name: count, dtype: int64

In [43]:
error_rows_food_why = food[food["reasoning"] == "video is missing, deleted, private, or unavailable"]

display(error_rows_food_why["video_id"].value_counts())

video_id
0r50VuyPmJM    1
41HWSuTc0zo    1
Name: count, dtype: int64

In [44]:
# 에러에 해당하는 이유가 "video is missing, deleted, private, or unavailable"인 경우를 제외한 데이터프레임을 생성합니다.
target_reason = "video is missing, deleted, private, or unavailable"

food_clean = food[food["reasoning"] != target_reason].copy()


In [45]:
food_clean.to_csv(
    "youtube_redirect_results_food_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# 병합하기

In [46]:
# 1) IT 병합
it_src = pd.read_csv("../../data/filtered_it_video_info.csv")
it_cls = pd.read_csv("youtube_redirect_results_it.csv", keep_default_na=False)  # "N/A"를 NaN으로 바꾸지 않음

it_merged = it_src.merge(
    it_cls,
    on="video_id",
    how="left",
    suffixes=("", "_redirect") # 
)

it_merged.to_csv(
    "filtered_it_with_redirect.csv",
    index=False,
    encoding="utf-8-sig"
)

In [47]:
# 2) FOOD 병합
food_src = pd.read_csv("../../data/filtered_food_video_info.csv")
food_cls = pd.read_csv("youtube_redirect_results_food_clean.csv", keep_default_na=False)

food_merged = food_src.merge(
    food_cls,
    on="video_id",
    how="left",
    suffixes=("", "_redirect")
)

food_merged.to_csv(
    "filtered_food_with_redirect.csv",
    index=False,
    encoding="utf-8-sig"
)


In [48]:
print("done")
print("IT:", len(it_src), "->", len(it_merged))
print("FOOD:", len(food_src), "->", len(food_merged))


done
IT: 8431 -> 8431
FOOD: 10852 -> 10852


# 병합된 코드 value count보기


In [49]:
df_food_merged = pd.read_csv("filtered_food_with_redirect.csv")
df_it_merged = pd.read_csv("filtered_it_with_redirect.csv")

In [50]:
display(df_food_merged.value_counts("verdict"))
display(df_it_merged.value_counts("verdict"))

verdict
shorts      5534
longform    5316
Name: count, dtype: int64

verdict
longform    5958
shorts      2473
Name: count, dtype: int64

In [51]:
# 1) 채널별 verdict 건수 집계
summary_food = pd.crosstab(df_food_merged["channel_title"], df_food_merged["verdict"])
summary_food = summary_food.sort_values(by=summary_food.columns.tolist(), ascending=False)
summary_it = pd.crosstab(df_it_merged["channel_title"], df_it_merged["verdict"])
summary_it = summary_it.sort_values(by=summary_it.columns.tolist(), ascending=False)

In [52]:
summary_food


verdict,longform,shorts
channel_title,,
농심 nongshim,427,205
CU [씨유튜브],416,554
GS25 l 이리오너라,351,707
동원TV,320,414
맛깔스튜디오 by롯데웰푸드,297,251
유사나헬스사이언스코리아,278,53
Coca-Cola Korea,210,136
매일유업 | Maeil,183,157
파리바게뜨TV,183,129


In [53]:
summary_it

verdict,longform,shorts
channel_title,,
SK브로드밴드_B tv,1734,111
Dell Technologies,979,222
삼성SDS,599,243
Microsoft Korea,438,59
빗썸 Bithumb Official,404,738
네이버 NAVER,375,219
네이버클라우드 l NAVER Cloud,309,10
카카오,290,345
kt cloud,219,8


In [54]:
# 2) 채널 내 verdict 비율(행 기준)
summary_ratio_food = pd.crosstab(
    df_food_merged["channel_title"], df_food_merged["verdict"], normalize="index"
).round(2)
summary_ratio_it = pd.crosstab(
    df_it_merged["channel_title"], df_it_merged["verdict"], normalize="index"
).round(2)

In [55]:
summary_ratio_food


verdict,longform,shorts
channel_title,,
BUDWEISER KOREA,0.56,0.44
CJ제일제당,0.27,0.73
CU [씨유튜브],0.43,0.57
Coca-Cola Korea,0.61,0.39
GS25 l 이리오너라,0.33,0.67
HITEJINRO,0.73,0.27
Samyangfoods삼양식품,0.42,0.58
Starbucks Korea,0.28,0.72
gongcha_korea,0.45,0.55


In [56]:
summary_ratio_it

verdict,longform,shorts
channel_title,,
Dell Technologies,0.82,0.18
Google Korea,0.54,0.46
LG CNS,0.48,0.52
Microsoft Korea,0.88,0.12
NOL,0.51,0.49
SK브로드밴드_B tv,0.94,0.06
kt M모바일,0.59,0.41
kt cloud,0.96,0.04
네이버 NAVER,0.63,0.37
